# 系统综述初筛助手

> 用 meta-catalog → meta-search → agentic-search 做 PRISMA-style 初筛

**场景**: 医学、生命科学、材料等领域的研究者需要做系统综述，第一步是 PRISMA-style 初筛：从海量文献中筛选出符合纳入标准的候选论文。

**使用接口**: meta-catalog, meta-search, agentic-search, content

**预估调用量**: ~30–80 次 API 调用 / 一次初筛任务

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx pandas
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 查询可用筛选字段

用 meta-catalog 确认数据库支持哪些筛选条件


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def get_catalog():
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(f"{BASE}/meta-catalog", headers=HEADERS)
        resp.raise_for_status()
        return resp.json()["fields"]

fields = asyncio.run(get_catalog())
for f in fields:
    print(f"{f['name']} ({f['type']}): operators={f['operators']}")


## Step 3: 广撒网检索

用 meta-search 按年份和关键词获取候选池


In [ ]:
import pandas as pd

async def broad_search(query: str, year_from: int, year_to: int, page_size: int = 100):
    """\u5e7f\u6492\u7f51: \u6309\u5e74\u4efd\u8303\u56f4\u68c0\u7d22\u6240\u6709\u5019\u9009\u6587\u732e"""
    all_results = []
    page = 1
    while True:
        async with httpx.AsyncClient(timeout=30) as client:
            resp = await client.post(
                f"{BASE}/meta-search", headers=HEADERS,
                json={
                    "query": query,
                    "filters": [
                        {"field": "publication_published_year", "operator": "FILTER_OP_GTE", "value": year_from},
                        {"field": "publication_published_year", "operator": "FILTER_OP_LTE", "value": year_to},
                    ],
                    "page": page, "page_size": page_size
                }
            )
            resp.raise_for_status()
            data = resp.json()
            all_results.extend(data["results"])
            if len(all_results) >= data["total_count"] or len(data["results"]) < page_size:
                break
            page += 1
    return all_results, data["total_count"]

results, total = asyncio.run(broad_search(
    "CAR-T cell therapy solid tumor clinical trial", 2019, 2024
))
print(f"Identification: {total} records found")


## Step 4: 语义精筛与纳入判断

用 agentic-search 对候选文献做语义相关性评分，筛选符合纳入标准的论文


In [ ]:
async def semantic_screen(query: str, top_k: int = 100):
    """\u8bed\u4e49\u7cbe\u7b5b: \u7528 agentic-search \u5bf9\u5019\u9009\u6587\u732e\u8bc4\u5206"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search", headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return resp.json()["hits"]

hits = asyncio.run(semantic_screen(
    "CAR-T cell therapy clinical trial solid tumor patients outcomes"
))

# \u6309\u76f8\u5173\u6027\u5206\u6570\u7b5b\u9009
screened = [h for h in hits if h["score"] >= 0.7]
print(f"Screening: {len(screened)} records (score >= 0.7)")

# \u8f93\u51fa PRISMA \u6d41\u7a0b\u6570\u636e
prisma = {
    "identification": total,
    "screening": len(screened),
    "included": len([h for h in screened if h["score"] >= 0.85])
}
print(f"\
PRISMA Flow: {prisma}")

# \u5bfc\u51fa CSV
df = pd.DataFrame(screened)
df.to_csv("screened_papers.csv", index=False)
print("Exported to screened_papers.csv")


## 注意事项

- 适合医学、生命科学、材料等高频系统综述场景
- meta-search 广撒网阶段可能需要分页拉取，注意 page_size 上限
- agentic-search 的 score 阈值建议根据领域调整（0.7–0.85）
- 完整 PRISMA 流程还需人工全文审阅，本案例覆盖自动化初筛部分
- 建议将筛选结果导出为 CSV 便于团队协作审阅


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
